# Unit 7 Risk Modelling Activity 2: Think Bayes, Chapters 1 and 2

Ian Stevens, Security and Risk Management, Unit 7.

My answers to the exercises at the end of chapters 1 (Probability) and 2 (Bayes's Theorem) of Downey's *Think Bayes* (2nd edn). The helper functions `prob`, `conditional` and `update` are Downey's, copied from the chapters so that the notebook runs on its own. Chapter 1 uses Downey's extract of the General Social Survey (GSS), `gss_bayes.csv`, which the first code cell downloads from the book's repository if it is not already in the folder.

In [1]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print('Downloaded ' + local)

download('https://github.com/AllenDowney/ThinkBayes2/raw/master/data/gss_bayes.csv')

In [2]:
import pandas as pd
from fractions import Fraction

gss = pd.read_csv('gss_bayes.csv')
len(gss)

49290

## Chapter 1: Probability

Downey's definitions: a probability is the fraction of a finite set, and a conditional probability is the fraction of a subset.

In [3]:
def prob(A):
    """Computes the probability of a proposition, A."""
    return A.mean()

def conditional(proposition, given):
    """Probability of A conditioned on given."""
    return prob(proposition[given])

banker = (gss['indus10'] == 6870)
female = (gss['sex'] == 2)
liberal = (gss['polviews'] <= 3)
democrat = (gss['partyid'] <= 1)
young = (gss['age'] < 30)
old = (gss['age'] >= 65)
conservative = (gss['polviews'] >= 5)

Before answering the exercises I checked that the data reproduce the figures printed in the chapter.

In [4]:
check = pd.DataFrame({
    'This notebook': [prob(banker), prob(female), prob(liberal), prob(democrat),
                      prob(young), prob(old), prob(conservative)],
    'Printed in chapter 1': [0.014769730168391155, 0.5378575776019476, 0.27374721038750255,
                             0.3662609048488537, 0.19435991073240008, 0.17328058429701765,
                             0.3419354838709677],
}, index=['banker', 'female', 'liberal', 'democrat', 'young', 'old', 'conservative'])
check['Match'] = (check.iloc[:, 0] - check.iloc[:, 1]).abs() < 1e-12
check

,This notebook,Printed in chapter 1,Match
banker,0.014770,0.014770,True
female,0.537858,0.537858,True
liberal,0.273747,0.273747,True
democrat,0.366261,0.366261,True
young,0.194360,0.194360,True
old,0.173281,0.173281,True
conservative,0.341935,0.341935,True


### Exercise 1.1

*Use `conditional` to compute: (a) the probability that a respondent is liberal, given that they are a Democrat; (b) the probability that a respondent is a Democrat, given that they are liberal.*

In [5]:
print(f"P(liberal | Democrat) = {conditional(liberal, given=democrat):.4f}")
print(f"P(Democrat | liberal) = {conditional(democrat, given=liberal):.4f}")

P(liberal | Democrat) = 0.3891
P(Democrat | liberal) = 0.5206

About 39% of Democrats are liberal, but about 52% of liberals are Democrats. The two differ because the groups are different sizes: P(liberal | Democrat) = P(liberal and Democrat) / P(Democrat), and there are more Democrats (37%) than liberals (27%) in the survey. Swapping the arguments is the error the chapter warns about, and it is the same error as reading P(alert | attack) as if it were P(attack | alert) (see the end of this notebook).

### Exercise 1.2: a variation of the Linda problem

*Which is more probable: Linda is a banker, or Linda is a banker and considers herself a liberal Democrat? Compute the probability that Linda is a banker given that she is female, and the probability that she is a banker and a liberal Democrat given that she is female.*

In [6]:
p_banker = conditional(banker, given=female)
p_banker_lib_dem = conditional(banker & liberal & democrat, given=female)
print(f"P(banker | female)                     = {p_banker:.4f}")
print(f"P(banker and liberal Democrat | female) = {p_banker_lib_dem:.4f}")
print(f"Ratio: {p_banker / p_banker_lib_dem:.1f} to 1")

P(banker | female)                     = 0.0212
P(banker and liberal Democrat | female) = 0.0023
Ratio: 9.2 to 1

The conjunction is about a tenth as likely. Adding a detail that makes the description feel more fitting can only make it less probable, which is Tversky and Kahneman's conjunction fallacy. A risk scenario behaves the same way: the more specific and vivid the chain of events (phishing email, then stolen credentials, then altered supplier records, then spoiled stock), the more convincing it sounds and the less likely it is, because each step is another conjunction.

### Exercise 1.3: young, old, liberal and conservative

*Young means under 30 and old means 65 or over. Compute: (a) the probability that a randomly chosen respondent is a young liberal; (b) the probability that a young person is liberal; (c) the fraction of respondents who are old conservatives; (d) the fraction of conservatives who are old.*

In [7]:
answers = pd.Series({
    '(a) P(young and liberal)  [conjunction]': prob(young & liberal),
    '(b) P(liberal | young)    [conditional]': conditional(liberal, given=young),
    '(c) P(old and conservative) [conjunction]': prob(old & conservative),
    '(d) P(old | conservative) [conditional]': conditional(old, given=conservative),
}, name='probability')
answers.round(4).to_frame()

,probability
(a) P(young and liberal) [conjunction],0.0658
(b) P(liberal | young) [conditional],0.3385
(c) P(old and conservative) [conjunction],0.0670
(d) P(old | conservative) [conditional],0.1960


(a) and (c) are conjunctions over the whole survey; (b) and (d) are conditional probabilities within a subgroup. Answer (d) is about 20%, under the 30% that Downey says would mean the arguments were the wrong way round. For comparison, P(conservative | old) is shown below; it answers a different question (what share of old people are conservative), and it is the one the quotation is really about.

In [8]:
print(f"P(conservative | old) = {conditional(conservative, given=old):.4f}")
print(f"P(liberal | young)     = {conditional(liberal, given=young):.4f}")

P(conservative | old) = 0.3867
P(liberal | young)     = 0.3385

Over the whole survey, 34% of young respondents describe themselves as liberal and 39% of old respondents as conservative, against 27% and 34% of all respondents, so the data lean the way the quotation suggests but give it nowhere near the force it claims. The dataset also pools surveys from 1974 onwards, so these are differences between generations as well as between ages.

## Chapter 2: Bayes's Theorem

Downey's `update` function turns a table of priors and likelihoods into posteriors.

In [9]:
def update(table):
    """Compute the posterior probabilities."""
    table['unnorm'] = table['prior'] * table['likelihood']
    prob_data = table['unnorm'].sum()
    table['posterior'] = table['unnorm'] / prob_data
    return prob_data

### Exercise 2.1: the trick coin

*One normal coin and one coin with heads on both sides. You pick one at random and see that one side is heads. What is the probability that you chose the trick coin?*

Hypotheses: normal coin or trick coin, equally likely. The data, seeing heads, has probability 1/2 under the normal coin and 1 under the trick coin.

In [10]:
coins = pd.DataFrame(index=['Normal', 'Trick'])
coins['prior'] = Fraction(1, 2)
coins['likelihood'] = Fraction(1, 2), 1
update(coins)
coins

,prior,likelihood,unnorm,posterior
Normal,1/2,1/2,1/4,1/3
Trick,1/2,1,1/2,2/3


**Answer: 2/3.** Seeing heads is twice as likely with the trick coin, so it doubles that coin's share of the probability.

### Exercise 2.2: two children

*Someone has two children. Asked whether either is a girl, they say yes. What is the probability that both are girls?*

Following the hint, start with four equally likely hypotheses for (older, younger): GG, GB, BG, BB. The answer 'yes' is certain under the first three and impossible under BB.

In [11]:
children = pd.DataFrame(index=['GG', 'GB', 'BG', 'BB'])
children['prior'] = Fraction(1, 4)
children['likelihood'] = 1, 1, 1, 0
update(children)
children

,prior,likelihood,unnorm,posterior
GG,1/4,1,1/4,1/3
GB,1/4,1,1/4,1/3
BG,1/4,1,1/4,1/3
BB,1/4,0,0,0


**Answer: 1/3.** The intuitive answer of 1/2 would be right for 'is the *older* child a girl?'. The question asked rules out only BB, which leaves three equally likely families, one of which is GG. The answer depends on exactly what was observed, not only on what is now known.

### Exercise 2.3: a variation of Monty Hall

*Monty always opens Door 2 if he can, and opens Door 3 only if he has to (because the car is behind Door 2). You choose Door 1. (a) If Monty opens Door 2, what is the probability the car is behind Door 3? (b) If Monty opens Door 3, what is the probability the car is behind Door 2?*

The likelihoods change because Monty's behaviour changes. (a) If the car is behind Door 1, Monty can open Door 2, so he does (likelihood 1). If it is behind Door 2 he cannot open it (0). If it is behind Door 3 he opens Door 2 (1).

In [12]:
monty_a = pd.DataFrame(index=['Door 1', 'Door 2', 'Door 3'])
monty_a['prior'] = Fraction(1, 3)
monty_a['likelihood'] = 1, 0, 1
update(monty_a)
monty_a

,prior,likelihood,unnorm,posterior
Door 1,1/3,1,1/3,1/2
Door 2,1/3,0,0,0
Door 3,1/3,1,1/3,1/2


(b) Monty opening Door 3 means he could not open Door 2, so the car must be behind Door 2. If it were behind Door 1 he would have opened Door 2 (likelihood 0); if behind Door 3 he could not open Door 3 (0).

In [13]:
monty_b = pd.DataFrame(index=['Door 1', 'Door 2', 'Door 3'])
monty_b['prior'] = Fraction(1, 3)
monty_b['likelihood'] = 0, 1, 0
update(monty_b)
monty_b

,prior,likelihood,unnorm,posterior
Door 1,1/3,0,0,0
Door 2,1/3,1,1/3,1
Door 3,1/3,0,0,0


**Answers: (a) 1/2, (b) 1.** When Monty opens Door 2 it tells us nothing to separate Doors 1 and 3, so switching gains nothing. When he opens Door 3, he gives the answer away. The lesson I take from this one is that the likelihood comes from a model of how the evidence was produced. Change the host's habits and the same observation means something different.

### Exercise 2.4: the M&M problem

*In 1994 a bag of plain M&M's was 30% brown, 20% yellow, 20% red, 10% green, 10% orange, 10% tan. In 1996 it was 24% blue, 20% green, 16% orange, 14% yellow, 13% red, 13% brown. One M&M is taken from each of two bags, one bag from 1994 and one from 1996. One is yellow and one is green. What is the probability that the yellow one came from the 1994 bag?*

Following the hint, the hypotheses describe *which bag is which*, and the data is the pair of colours:

* **A**: the yellow came from the 1994 bag and the green from the 1996 bag: likelihood 0.20 x 0.20
* **B**: the yellow came from the 1996 bag and the green from the 1994 bag: likelihood 0.14 x 0.10

In [14]:
mm = pd.DataFrame(index=['A: yellow from 1994', 'B: yellow from 1996'])
mm['prior'] = Fraction(1, 2)
mm['likelihood'] = Fraction(20, 100) * Fraction(20, 100), Fraction(14, 100) * Fraction(10, 100)
update(mm)
mm

,prior,likelihood,unnorm,posterior
A: yellow from 1994,1/2,1/25,1/50,20/27
B: yellow from 1996,1/2,7/500,7/1000,7/27


In [15]:
print(f"P(yellow came from the 1994 bag) = {mm.loc['A: yellow from 1994', 'posterior']} "
      f"= {float(mm.loc['A: yellow from 1994', 'posterior']):.3f}")

P(yellow came from the 1994 bag) = 20/27 = 0.741

**Answer: 20/27, about 0.74.**

## Applying chapter 2 to a security question

The same Bayes table answers a question that matters in security operations: when an alert fires, how likely is it that an attack is really happening? The figures below are **illustrative, not measured**: a detection rule that catches 95% of real attacks, raises a false alarm on 2% of benign events, and a base rate of 1 malicious event in 1,000.

In [16]:
alert = pd.DataFrame(index=['Malicious', 'Benign'])
alert['prior'] = 0.001, 0.999
alert['likelihood'] = 0.95, 0.02     # P(alert | malicious), P(alert | benign)
p_alert = update(alert)
print(f"P(alert) = {p_alert:.4f}")
alert.round(4)

P(alert) = 0.0209

,prior,likelihood,unnorm,posterior
Malicious,0.001,0.95,0.001,0.0454
Benign,0.999,0.02,0.020,0.9546


Even with a 95% detection rate, fewer than 1 alert in 20 is a real attack, because benign events vastly outnumber malicious ones. This is exercise 1.1 again: P(alert | attack) is high, but P(attack | alert) is low. A risk assessment that rates a control on its detection rate alone, without the base rate, will overstate what the control delivers and understate the analyst time spent on false alarms.

## References

Downey, A.B. (2021) *Think Bayes: Bayesian statistics in Python*. 2nd edn. Sebastopol, CA: O'Reilly Media. Available at: https://allendowney.github.io/ThinkBayes2/

Tversky, A. and Kahneman, D. (1983) 'Extensional versus intuitive reasoning: the conjunction fallacy in probability judgment', *Psychological Review*, 90(4), pp. 293-315. Available at: https://doi.org/10.1037/0033-295X.90.4.293